# Student Lighthouse — Machine Learning Analysis

**Problem:** The student success service can only contact 200 students per semester. Which ones should they call first?
**Target:** Binary classification (Dropout = 1, Graduate/Enrolled = 0).
**Cost of error:** A false negative means a student drops out without help. A false positive means a wasted call.

In [1]:
# On importe nos propres classes depuis le package `lighthouse`
import sys
import os
sys.path.append(os.path.abspath("..")) # Permet de trouver le package

import pandas as pd
from lighthouse.data import DatasetLoader
from lighthouse.models import Trainer
from lighthouse.evaluate import evaluate_final_model, get_baseline_scores
from lighthouse.features import NUMERICAL_COLS, CATEGORICAL_COLS

## 1. Data Loading and Exploration
We use the `DatasetLoader` class from our package. It handles the CSV loading, removes the Semester 2 columns (to prevent temporal leakage), and maps the target.

In [2]:
# On charge les données via notre package
loader = DatasetLoader()
df = loader.load_and_clean()

print(f"Dataset shape: {df.shape}")
print("Target distribution:")
print(df["target"].value_counts(normalize=True).round(2))

# On sépare le train et le test (stratifié)
X_train, X_test, y_train, y_test = loader.split(df)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

Dataset shape: (4424, 31)
Target distribution:
target
0    0.68
1    0.32
Name: proportion, dtype: float64
Train size: 3539 | Test size: 885


## 2. Baseline
Before training any model, we check the performance of a dummy classifier that always predicts the majority class.

In [3]:
# On évalue la baseline
baseline_scores = get_baseline_scores(X_train, y_train, X_test, y_test)

## 3. Model Comparison
We compare 3 models under the same cross-validation, using Average Precision (PR-AUC) because the classes are imbalanced.

In [4]:
# On compare les modèles via notre classe Trainer
trainer = Trainer()
results = trainer.compare_models(X_train, y_train)

## 4. Hyperparameter Tuning
We tune at least one hyperparameter per model using `GridSearchCV`.

In [5]:
# On lance le GridSearch pour trouver le meilleur modèle
best_model_info = trainer.tune_and_select(X_train, y_train)

Logistic Regression : Meilleur score = 0.8556 | Params = {'model__C': 1.0}
k-NN : Meilleur score = 0.7436 | Params = {'model__n_neighbors': 11}
Linear SVM : Meilleur score = 0.8555 | Params = {'model__C': 0.1}

--> Modèle retenu : Logistic Regression (Score : 0.8556)


## 5. Decision Threshold
The service can only contact 200 students. We calculate the exact threshold that selects the top 200 students on out-of-fold predictions.

In [6]:
# On calcule le seuil pour une capacité de 200
threshold = trainer.calculate_threshold(X_train, y_train, capacity=200)

Seuil pour contacter 200 étudiants : 0.9927


## 6. Final Test Evaluation
We train the final model on the whole training set, and evaluate it **once** on the test set.

In [7]:
# On entraîne le modèle final sur tout le train set
final_pipeline = trainer.fit(X_train, y_train)

# On évalue sur le test set (une seule fois !)
final_metrics = evaluate_final_model(final_pipeline, X_test, y_test, threshold)


--- FINAL MODEL EVALUATION ---
Average Precision : 0.8727
ROC-AUC           : 0.9120
Precision @ Top 50 : 0.9800
Confusion Matrix  :
[[600   1]
 [227  57]]


## 7. Interpretation
Which features drive the model's decisions?

In [8]:
# On affiche les 10 coefficients les plus importants
preprocessor = final_pipeline.named_steps["preprocessor"]
classifier = final_pipeline.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()

import pandas as pd
coefs = pd.DataFrame({
    "Feature": [f.replace("num__", "").replace("cat__", "") for f in feature_names],
    "Coefficient": classifier.coef_[0]
})

# On trie par influence absolue
coefs["Abs"] = coefs["Coefficient"].abs()
coefs = coefs.sort_values("Abs", ascending=False).head(10)

print(coefs[["Feature", "Coefficient"]])

                                 Feature  Coefficient
6    Curricular units 1st sem (approved)    -2.327446
175              Mother's occupation_191    -1.797633
148                Mother's occupation_0     1.421754
130            Father's qualification_22     1.326930
190               Father's occupation_90     1.313864
51                           Course_9853     1.286939
93              Mother's qualification_4    -1.271665
229            Tuition fees up to date_1    -1.229140
228            Tuition fees up to date_0     1.209366
89                       Nacionality_109     1.165430


## 8. Limitations and Risks

### Limitations of the model
- **The "Enrolled" ambiguity:** We chose to group "Enrolled" with "Graduate" (no dropout). However, an enrolled student is not guaranteed to graduate. The model might under-estimate the risk for students who are currently struggling but have not officially dropped out yet.
- **The cost of removing S2 data:** We dropped all 2nd-semester features to prevent temporal leakage. This is the honest choice for real-world deployment, but it comes at a price: our model's performance is lower than it would be if we kept them. We accepted this trade-off because a model that predicts *after* the results are known is useless for early intervention.
- **No qualitative data:** The dataset only contains administrative and academic metrics. It cannot capture a student's mental health, motivation, or personal struggles. A student with perfect grades might still drop out due to personal reasons, and our model would miss it completely.

### Risks for real people
- **Stigmatization:** If an advisor treats the "High Risk" label as a certainty, they might unconsciously treat the student differently. The Student Card must be presented as a *conversation starter*, not a verdict. The human advisor is the one making the final decision.
- **The 200-slot cutoff:** Because we set a strict threshold for 200 students, a student ranked 201st is completely ignored, even if their risk score is 0.9926 (just below our 0.9927 threshold). This is a harsh reality of limited resources: our model optimizes for the quota, but it means some at-risk students will fall through the cracks.
- **Fairness across demographics:** We included features like age, gender, and nationality. While these can be relevant, the model could inadvertently learn biases from historical data. For example, if older students historically dropped out more often, the model might over-flag them, leading to unfair treatment. A proper fairness audit (comparing error rates across genders) would be mandatory before using this in production.